In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

In [2]:
db_path = r"D:\Projects\Customer_Retention\mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{db_path.replace('\\', '/')}")

df_rfm = pd.read_parquet(r'D:\\Projects\\Customer_Retention\data\\02_rfm_features.parquet')

df_raw = pd.read_parquet(r'D:\\Projects\\Customer_Retention\data\\01_cleaned_retail.parquet')


In [ ]:

model_name = "Best_Churn_Predictor"
model_uri = f"models:/{model_name}/latest"
best_model = mlflow.sklearn.load_model(model_uri)

features = ['Frequency', 'Monetary', 'F_Score', 'M_Score']


df_rfm['Churn_Probability'] = best_model.predict_proba(df_rfm[features])[:, 1]


In [ ]:
clv_summary = summary_data_from_transaction_data(
    df_raw, 'Customer ID', 'InvoiceDate', 'Revenue',
    observation_period_end=df_raw['InvoiceDate'].max()
)
clv_summary = clv_summary[clv_summary['frequency'] > 0]

bgf = BetaGeoFitter(penalizer_coef=0.01).fit(clv_summary['frequency'], clv_summary['recency'], clv_summary['T'])
ggf = GammaGammaFitter(penalizer_coef=0.01).fit(clv_summary['frequency'], clv_summary['monetary_value'])

clv_summary['6_Month_CLV'] = ggf.customer_lifetime_value(
    bgf, clv_summary['frequency'], clv_summary['recency'], clv_summary['T'], clv_summary['monetary_value'],
    time=6, discount_rate=0.01
)

master_df = df_rfm.merge(clv_summary[['6_Month_CLV']], left_on='Customer ID', right_index=True, how='left')
master_df['6_Month_CLV'] = master_df['6_Month_CLV'].fillna(0) # For non-repeaters

print("Master Action Table Created!")
master_df[['Customer ID', 'Segment_Label', 'Churn_Probability', '6_Month_CLV']].head()